# Lesson 1. The state of a qubit

**Goal:** understand what a qubit is physically, what its state is mathematically, and why these are two different things.

**Notation:** we write in the language of linear algebra. The standard basis of $\mathbb{C}^2$ is $e_0 = (1, 0)$ and $e_1 = (0, 1)$. Physicists write them $|0\rangle$ and $|1\rangle$; we use that notation only as labels, because libraries (PennyLane, Qiskit) report results that way.

## 1. A qubit is a physical system

A **two-level system** is a quantum system in which exactly two states with different energies matter:

- the **ground state** (lowest energy), which by convention we call $e_0$;
- the **excited state**, which we call $e_1$.

The names "0" and "1" are a convention. Physically they are just "lower" and "higher" energy.

Some systems have exactly two states (electron spin, photon polarisation). Most have more, and we **pick the two lowest** and make sure the system never reaches the third.

### The transmon: a superconducting qubit (IBM Heron, Google Willow)

It is an electrical circuit on a chip: a capacitor and a **Josephson junction** (two superconductors separated by a thin insulating layer). Once cooled, it behaves like an artificial atom with discrete energy levels.

The key point is that the levels are **unevenly** spaced (anharmonicity). The $0 \to 1$ transition has a different frequency from $1 \to 2$, so a microwave pulse tuned to $0 \leftrightarrow 1$ barely excites level 2. The anharmonicity is only about −300 MHz, a few percent of the qubit frequency, so pulses are shaped (e.g. DRAG) to suppress this leakage. If the levels were evenly spaced (an ordinary harmonic oscillator), every pulse driving $0 \to 1$ would also drive $1 \to 2$, $2 \to 3$ and so on, and two levels could not be isolated.

Control is not perfect: sometimes the state leaks into level 2 (*leakage*). This is one source of errors.

### Why cool to about 15 mK?

For three reasons:

1. **Superconductivity.** The aluminium in the junctions becomes superconducting only below about 1.2 K. Without it there is no qubit.
2. **A known starting point.** The $0 \to 1$ energy (about 5 GHz, i.e. about 0.24 K) must be much larger than thermal energy. At 15 mK the environment is more than ten times "too weak" to excite the qubit by accident, so the qubit relaxes to $e_0$ and stays there.
3. **Quiet during computation.** Every accidental excitation is an error. Low temperature extends the time a qubit keeps its state.

### How do we prepare the initial state $e_0 \otimes \dots \otimes e_0$?

- **Waiting:** an excited qubit relaxes to $e_0$ by itself (relaxation, time $T_1$ of the order of 100 µs). Waiting a few $T_1$ is enough.
- **Active reset (faster):** measure the qubit; if the result is 1, apply an X gate.

In practice the qubit is in $e_0$ with probability about 99%, not 100%. This is part of the so-called SPAM errors (*state preparation and measurement*). Every other initial state is built from $e_0$ with gates.

## 2. The state of a qubit is a unit vector in $\mathbb{C}^2$

$$v = \alpha e_0 + \beta e_1 = (\alpha, \beta), \qquad \alpha, \beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

- $\alpha$ and $\beta$ are **amplitudes**: the coordinates of the state vector in the basis $\{e_0, e_1\}$. They are complex and they are not probabilities.
- The normalisation condition just says $\|v\| = 1$.
- Watch the modulus: $|i|^2 = 1$, not $i^2 = -1$.
- A state is defined up to a **global phase**: $v$ and $e^{i\gamma} v$ describe the same physical state (details in lesson 3). Formally a state is a ray, not a vector.

The pair $(\alpha, \beta)$ is four real numbers regardless of notation (Cartesian $a + bi$ or polar $r e^{i\varphi}$). The notation changes which four numbers you see, not how many there are.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
GREEN, DARK, LIGHT, GREY, ORANGE = "#44B749", "#1A8A1F", "#96E199", "#8A8A8A", "#E08A00"

e0 = np.array([1, 0], dtype=complex)   # label |0⟩
e1 = np.array([0, 1], dtype=complex)   # label |1⟩

def norm(v):
    return np.sqrt(np.sum(np.abs(v) ** 2))

def show(v, n=None):
    """Print a state vector with bit labels (label = position written in binary)."""
    v = np.asarray(v)
    n = n or int(np.log2(len(v)))
    for k, a in enumerate(v):
        if abs(a) > 1e-12:
            print(f"  position {k} ({format(k, f'0{n}b')}): amplitude {np.round(a, 3)}, P = {abs(a)**2:.3f}")

# Which vectors can be the state of a qubit? Norm 1 is all it takes.
candidates = {
    "(1/2, 1/2)":       np.array([1/2, 1/2]),
    "(1/√2, i/√2)":     np.array([1, 1j]) / np.sqrt(2),
    "(3/5, 4/5)":       np.array([3/5, 4/5]),
    "(1, 1)":           np.array([1, 1]),
}
for name, v in candidates.items():
    ok = np.isclose(norm(v), 1)
    print(f"{name:>14}: ||v||² = {norm(v)**2:.3f}  ->  {'a state' if ok else 'NOT a state'}")

    (1/2, 1/2): ||v||² = 0.500  ->  NOT a state
  (1/√2, i/√2): ||v||² = 1.000  ->  a state
    (3/5, 4/5): ||v||² = 1.000  ->  a state
        (1, 1): ||v||² = 2.000  ->  NOT a state

## 3. Three things not to confuse

> **This was tricky:** the words "qubit", "state" and "outcome" blur easily. They are three different objects.

| | Qubit | State | Measurement outcome |
|---|---|---|---|
| What it is | a physical system (transmon, ion, photon) | a vector $(\alpha, \beta) \in \mathbb{C}^2$ | the number 0 or 1 |
| Is it random | no | **no**, it is definite | **yes** |
| Does it change | no | yes: gates and measurement change it | it only appears at measurement |
| Can it be read out | – | not directly | yes, it is what we see |

**Amplitudes are not a property of the qubit like mass.** For three reasons:

1. **The state changes:** the same qubit is in $e_0$ at one moment, in $(e_0 + e_1)/\sqrt{2}$ after a gate, and in $e_0$ or $e_1$ after a measurement.
2. **Coordinates depend on the basis:** the same state has different $\alpha, \beta$ in another basis.
3. **They cannot be read out:** one measurement gives one bit, not $\alpha$ or $\beta$.

A good intuition: the qubit is the "container", and the state is its current content, described in a chosen coordinate system.

**Before measurement the qubit is not "really" 0 or 1.** It is in the state $(\alpha, \beta)$, which is neither zero nor one. Measurement does not read a pre-existing value, it creates one (lesson 2), and the experiment in lesson 3 shows that a superposition can be told apart from a "hidden 0 or 1". That no hidden-value model works at all is the content of Bell's theorem (lesson 7).

## 4. Two different "circuits"

| | Superconducting circuit | Quantum circuit |
|---|---|---|
| What it is | a physical element on the chip | a program: a description of a computation |
| Corresponds to | **one qubit** | **a whole algorithm** |
| Made of | a capacitor and a Josephson junction | wires (qubits) and gates in order |
| Where it lives | in the fridge, at 15 mK | in code (`qml.qnode`, `QuantumCircuit`) |

The word *circuit* in libraries comes from logic diagrams. The same program runs on a simulator (where there are no physical circuits at all, only matrix multiplication) and on a chip. A 3-qubit program on IBM Heron will usually take 3 of the 156 physical qubits, chosen by the transpiler. If the chosen qubits are not directly connected, the transpiler adds SWAP gates through other qubits. These are **physical**, noisy qubits, not logical, error-corrected ones.

## Summary

> A qubit is a physical two-level system, cooled so that it is superconducting and starts in its ground state. We describe its state with a unit vector $v = (\alpha, \beta) \in \mathbb{C}^2$, up to a global phase. The amplitudes $\alpha, \beta$ are the coordinates of this vector in the basis $\{e_0, e_1\}$, not fixed properties of the qubit.

## Exercises

1. Which of $(1/2, 1/2)$, $(1/\sqrt{2}, i/\sqrt{2})$, $(3/5, 4/5)$, $(1, 1)$ can be the state of a qubit? (Check the cell above: there are **two**.)
2. State $(\sqrt{3}/2, 1/2)$. What are $|\alpha|^2$ and $|\beta|^2$?
3. Why can a transmon, which has many energy levels, be used as a qubit? What would happen with evenly spaced levels?
4. A colleague says: "Our circuit has 3 qubits, so on IBM Heron it will take 3 of the 156 superconducting circuits". What is correct here, and what is a shortcut?